In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
import os


FILE_FLIGHTS = 'Final Aviation Cleaning.csv'
FILE_WEATHER = 'Weather_Clean_2026_Ready.csv'
FILE_AIRPORTS = 'airports_clean_dim (1).csv'

def run_analytics_engine():
    print("Initializing Aviation Analytics Engine...")


    for f in [FILE_FLIGHTS, FILE_WEATHER, FILE_AIRPORTS]:
        if not os.path.exists(f):
            raise FileNotFoundError(f"Critical Error: The file '{f}' was not found in the current directory. Please upload it to continue.")


    flights = pd.read_csv('/content/Final Aviation Cleaning.csv')
    weather = pd.read_csv('/content/Weather_Clean_2026_Ready.csv')
    airports = pd.read_csv('/content/airports_clean_dim (1).csv')


    flights['Flight Date'] = pd.to_datetime(flights['Flight Date'])
    weather['date'] = pd.to_datetime(weather['date'])


    df = pd.merge(flights, weather, left_on=['Departure IATA', 'Flight Date'],
                  right_on=['iata_code', 'date'], how='inner')

    df = pd.merge(df, airports[['iata_code', 'elevation_ft', 'type']],
                  left_on='Departure IATA', right_on='iata_code', suffixes=('', '_dim'))


    df['Wind_Band'] = pd.cut(df['wind_speed'], bins=np.arange(0, 75, 5))
    wind_wall = df.groupby('Wind_Band', observed=False).agg(
        Average_Delay=('Departure Delay_capped', 'mean'),
        Flight_Count=('Departure IATA', 'count')
    ).reset_index()
    wind_wall['Wind_Band'] = wind_wall['Wind_Band'].astype(str)

    avg_global_delay = df['Departure Delay_capped'].mean()
    wind_wall['Delay_Diff'] = wind_wall['Average_Delay'].diff()
    crit_mask = wind_wall['Delay_Diff'] > (avg_global_delay * 0.5)
    critical_threshold = wind_wall.loc[crit_mask, 'Wind_Band'].iloc[0] if crit_mask.any() else "N/A"


    X = df[['wind_speed']].fillna(0)
    y = df['Departure Delay_capped']
    reg = LinearRegression().fit(X, y)

    weather_elasticity = pd.DataFrame({
        'Wind_Speed': np.arange(0, 61),
        'Predicted_Delay': reg.predict(np.arange(0, 61).reshape(-1, 1))
    })
    elasticity_coeff = reg.coef_[0]
    r_squared = reg.score(X, y)


    pareto = df.groupby('Route').agg(Total_Delay=('Departure Delay_capped', 'sum')).sort_values('Total_Delay', ascending=False).reset_index()
    total_d = pareto['Total_Delay'].sum()
    pareto['Delay_Percentage'] = (pareto['Total_Delay'] / total_d) * 100
    pareto['Cumulative_Percentage'] = pareto['Delay_Percentage'].cumsum()


    connections = df.groupby('Departure IATA')['Arrival IATA'].nunique()
    vol = df['Departure IATA'].value_counts()
    network_fragility = pd.DataFrame({
        'Airport': connections.index,
        'Connections': connections.values,
        'Flight_Volume': vol.loc[connections.index].values
    }).reset_index(drop=True)
    network_fragility['Fragility_Score'] = (network_fragility['Connections'] / network_fragility['Connections'].max() * 0.6) + \
                                           (network_fragility['Flight_Volume'] / network_fragility['Flight_Volume'].max() * 0.4)


    vulner = df.groupby('Departure IATA').agg(
        Avg_Delay=('Departure Delay_capped', 'mean'),
        Wind_Exposure=('wind_speed', 'max'),
        Elevation=('elevation_ft', 'first')
    ).reset_index()
    scaler = MinMaxScaler()
    vulner[['Delay_Score', 'Wind_Score', 'Elevation_Score']] = scaler.fit_transform(vulner[['Avg_Delay', 'Wind_Exposure', 'Elevation']].fillna(0))
    vulner['Vulnerability_Index'] = vulner[['Delay_Score', 'Wind_Score', 'Elevation_Score']].mean(axis=1)


    df['Compression_Value'] = df['Departure Delay_capped'] - df['Arrival Delay_capped']
    schedule_compression = df.groupby('Route').agg(Compression_Opportunity=('Compression_Value', 'mean')).reset_index()

    df['Elevation_Group'] = np.where(df['elevation_ft'] <= 2000, 'Low Elevation', 'High Elevation')
    elevation_penalty = df.groupby('Elevation_Group').agg(Average_Delay=('Departure Delay_capped', 'mean'), Average_Recovery=('Compression_Value', 'mean')).reset_index()


    w_bins = [0, 10, 20, 35, 100]
    w_labels = ['Low Wind', 'Medium Wind', 'High Wind', 'Extreme Wind']
    df['Weather_Cohort'] = pd.cut(df['wind_speed'], bins=w_bins, labels=w_labels)
    weather_cohorts = df.groupby('Weather_Cohort', observed=False).agg(Average_Delay=('Departure Delay_capped', 'mean'), Flight_Count=('Departure IATA', 'count')).reset_index()

    e_bins = [-100, 1000, 3000, 15000]
    e_labels = ['Low Elevation', 'Medium Elevation', 'High Elevation']
    df['Elevation_Cohort'] = pd.cut(df['elevation_ft'], bins=e_bins, labels=e_labels)
    elevation_cohorts = df.groupby('Elevation_Cohort', observed=False).agg(Average_Delay=('Departure Delay_capped', 'mean'), Flight_Count=('Departure IATA', 'count')).reset_index()


    whatif_highwind = df[df['wind_speed'] > 25].groupby('Departure IATA').agg(Current_Delay=('Departure Delay_capped', 'mean')).reset_index()
    if not whatif_highwind.empty:
        whatif_highwind['Projected_Delay'] = whatif_highwind['Current_Delay'] * 0.8


    wsf_value = df['wind_speed'].corr(df['Departure Delay_capped'])
    weather_sensitivity_factor = pd.DataFrame({
        'Metric': ['Weather Sensitivity Factor'],
        'Value': [round(wsf_value,4)]
    })


    high_wind_df = df[df['wind_speed'] > 25]
    avg_delay_high_wind = high_wind_df['Departure Delay_capped'].mean()
    resilience_index = max(0, 100 * (1 - (avg_delay_high_wind / 60)))
    resilience_sheet = pd.DataFrame({
        'Metric': ['Resilience Index'],
        'Value': [round(resilience_index,2)]
    })


    route_optimization = df.groupby('Route').agg(Current_Delay=('Departure Delay_capped','mean')).reset_index()
    route_optimization['Optimized_Delay'] = route_optimization['Current_Delay'] * 0.85
    route_optimization['Savings'] = route_optimization['Current_Delay'] - route_optimization['Optimized_Delay']
    route_optimization = route_optimization.sort_values('Savings', ascending=False)

    high_elevation_priority = df[df['elevation_ft'] > 2000].groupby('Departure IATA').agg(Current_Delay=('Departure Delay_capped','mean')).reset_index()
    high_elevation_priority['Projected_Delay'] = high_elevation_priority['Current_Delay'] * 0.80
    high_elevation_priority['Improvement'] = high_elevation_priority['Current_Delay'] - high_elevation_priority['Projected_Delay']


    top_vulnerable_airport = vulner.loc[vulner['Vulnerability_Index'].idxmax(), 'Departure IATA']
    top_fragile_airport = network_fragility.loc[network_fragility['Fragility_Score'].idxmax(), 'Airport']
    top_delay_route = pareto.loc[pareto['Total_Delay'].idxmax(), 'Route']

    executive_summary = pd.DataFrame({
        'Metric': [
            'Critical Wind Threshold',
            'Weather Sensitivity Factor',
            'Resilience Index',
            'Elasticity Coefficient',
            'Model R Squared',
            'Most Vulnerable Airport',
            'Most Fragile Airport',
            'Highest Delay Route'
        ],
        'Value': [
            critical_threshold,
            round(wsf_value,4),
            round(resilience_index,2),
            round(elasticity_coeff,4),
            round(r_squared,4),
            top_vulnerable_airport,
            top_fragile_airport,
            top_delay_route
        ]
    })


    with pd.ExcelWriter('aviation_analytics_output.xlsx') as writer:
        df.to_excel(writer, sheet_name='Flights_Final', index=False)
        wind_wall.to_excel(writer, sheet_name='Wind_Velocity_Wall', index=False)
        weather_elasticity.to_excel(writer, sheet_name='Weather_Elasticity', index=False)
        pareto.to_excel(writer, sheet_name='Delay_Pareto', index=False)
        network_fragility.to_excel(writer, sheet_name='Network_Fragility', index=False)
        vulner.to_excel(writer, sheet_name='Airport_Vulnerability', index=False)
        elevation_penalty.to_excel(writer, sheet_name='Elevation_Penalty', index=False)
        weather_cohorts.to_excel(writer, sheet_name='Weather_Cohorts', index=False)
        elevation_cohorts.to_excel(writer, sheet_name='Elevation_Cohorts', index=False)
        whatif_highwind.to_excel(writer, sheet_name='WhatIf_HighWind', index=False)
        weather_sensitivity_factor.to_excel(writer, sheet_name='Weather_Sensitivity_Factor', index=False)
        resilience_sheet.to_excel(writer, sheet_name='Resilience_Index', index=False)
        route_optimization.to_excel(writer, sheet_name='Route_Optimization', index=False)
        high_elevation_priority.to_excel(writer, sheet_name='High_Elevation_Priority', index=False)
        executive_summary.to_excel(writer, sheet_name='Executive_Summary', index=False)
        schedule_compression.to_excel(
    writer,
    sheet_name='Schedule_Compression',
    index=False
)

    print("Success: Workbook 'aviation_analytics_output.xlsx' generated.")
    return critical_threshold, elasticity_coeff, r_squared

try:

    crit_wind, coeff, r2 = run_analytics_engine()


    print(f"Critical Wind Threshold: {crit_wind}")
    print(f"Elasticity (Min/Knot): {coeff:.2f}")
    print(f"Model R-Squared: {r2:.4f}")
except FileNotFoundError as e:
    print(e)

Initializing Aviation Analytics Engine...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


Success: Workbook 'aviation_analytics_output.xlsx' generated.
Critical Wind Threshold: (25, 30]
Elasticity (Min/Knot): 0.20
Model R-Squared: 0.0147
